In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )

print(PROJECT_ROOT)

/home/ubuntu/Projects/thesis-code


In [7]:
from src.baseline.dataset import COdeBaselineDataset
from src.baseline.transforms import get_image_transform
from src.baseline import config


dataset = COdeBaselineDataset(
    csv_path=config.DATASET_PATH,
    split="train",
    image_root=config.IMAGE_ROOT,
    transform=get_image_transform(),
)


sample = dataset[0]


print(sample["checkup_id"])
print(len(sample["images"]))
print(sample["images"][0].shape)
print(sample["labels"].shape)

train: 6129 samples
0001-001
1
torch.Size([3, 224, 224])
torch.Size([13])


# Thesis Note 05.2

# Baseline Experiment B — Six-Label Radiograph-only Fine-tuned Classification

## Overview

This experiment represents the official radiograph-only baseline for the six-label COde classification benchmark.

The objective is to establish a clean and reproducible unimodal baseline using radiographic images alone before introducing photographs, clinical text, and multimodal fusion.

The six-label benchmark was constructed from the previously finalized 13-label patient-level dataset while preserving the original patient identities, visits, modalities, and leakage-safe split assignments.

The experiment evaluates whether radiographic images alone contain useful information for automated multi-label dental diagnosis classification.

---

# 1. Experiment Objective

The goal of this experiment is to train and evaluate a radiograph-only deep learning classifier for six-label dental diagnosis prediction.

The model receives only radiographic images from each patient visit and predicts the following six diagnostic categories:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth Loss
6. Tooth Structure Loss

The experiment answers the following research question:

> How effective are radiographic images alone for automated multi-label dental diagnosis classification on the six-label COde benchmark?

---

# 2. Six-Label Benchmark Construction

The six-label benchmark was derived from the finalized 13-label patient-level dataset.

The original 13-label representation was converted into six benchmark labels.

The malocclusion label combines the three original malocclusion categories:

- Class I Malocclusion
- Class II Malocclusion
- Class III Malocclusion

The remaining selected conditions are retained as individual labels.

The final label set is:

- label_caries
- label_gingivitis
- label_malocclusion
- label_pulpitis
- label_tooth_loss
- label_tooth_structure_loss

The resulting dataset was validated successfully.

Final dataset:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

---

# 3. Dataset Configuration

Dataset:

COde Dataset

Task:

Six-label multi-label dental diagnosis classification

Number of labels:

6

Total visits:

8,775

Total patients:

4,800

The six-label dataset preserves the previously established patient-level split.

---

# 4. Patient-Level Split

The experiment uses the previously validated patient-level split.

Split strategy:

Patient-level split

Seed:

42

Train      : 3,360 patients / 6,129 visits
Validation :   720 patients / 1,330 visits
Test       :   720 patients / 1,316 visits

The patient-level split remains authoritative and was not modified during six-label benchmark construction.

Previous leakage auditing confirmed that the split is leakage-safe.

The six-label construction changes only the label representation and does not introduce a new patient split.

---

# 5. Label Coverage

The six-label benchmark contains fewer labeled visits than the original 13-label benchmark because a visit is considered labeled only when at least one of the six selected labels is positive.

Split coverage:

Train      : 4,649 / 6,129 labeled visits — 75.85%
Validation :   988 / 1,330 labeled visits — 74.29%
Test       : 1,004 / 1,316 labeled visits — 76.29%

This reduction in coverage is expected because the six-label benchmark intentionally focuses on a subset of the reconstructed diagnostic categories.

---

# 6. Input Modality

This experiment uses only radiographic images.

Used modality:

Radiographs

Excluded modalities:

Photographs

Clinical text

The input pipeline is:

Radiographs
    |
    v
ResNet50 Encoder
    |
    v
Mean Feature Aggregation
    |
    v
Classification Head
    |
    v
Six-label Prediction

Only visits containing at least one radiograph are included in the radiograph-only experiment.

---

# 7. Model Architecture

## Image Encoder

Backbone:

ResNet50

Initialization:

ImageNet pretrained weights

Training strategy:

Full encoder fine-tuning

The encoder parameters are updated during training rather than being frozen.

---

# 8. Variable-Length Radiograph Aggregation

Each dental visit may contain a variable number of radiographs.

Each radiograph is independently processed by the ResNet50 encoder and the resulting feature vectors are aggregated using mean pooling.

Architecture:

Radiograph 1 ----\
Radiograph 2 ----- ResNet50 ---- Mean Pooling ---- Classifier
Radiograph N ----/

Mathematically:

z = (1/N) Σ f(x_i)

where:

- x_i represents the i-th radiograph
- f(.) represents the ResNet50 encoder
- z represents the visit-level image representation

This produces a fixed-dimensional representation regardless of the number of radiographs associated with a visit.

---

# 9. Training Configuration

Modality:

Radiograph-only

Training samples:

2,972

Validation samples:

642

Batch size:

16

Number of epochs:

20

Loss function:

BCEWithLogitsLoss

Task formulation:

Multi-label classification

Learning rate:

1e-5

Weight decay:

1e-4

Encoder:

ResNet50

Pretrained:

ImageNet

Encoder:

Fine-tuned

Device:

CUDA

---

# 10. Training Results

The model showed substantially stronger learning behavior than the earlier 13-label radiograph-only experiment.

During the first epoch:

Train Loss:

0.4574

Validation Macro F1:

0.0913

Validation Micro F1:

0.4042

Validation AUROC:

0.5633

By epoch 12:

Validation Macro F1:

0.1503

Validation Micro F1:

0.5238

Validation AUROC:

0.7160

The best validation Macro F1 during training was approximately:

0.1830

at epoch 19.

The validation AUROC reached approximately:

0.7160

during training.

The training loss continuously decreased from:

0.4574

to:

0.1189

by epoch 20.

However, validation loss and threshold-dependent metrics fluctuated, indicating that the model began to overfit the training data during later epochs.

---

# 11. Threshold Optimization

Because this is a multi-label classification problem with substantially different label prevalences, a fixed probability threshold of 0.5 is not necessarily optimal for every diagnosis.

Therefore, label-specific thresholds were optimized on the validation split.

The procedure was:

1. Train the model using the training split.
2. Generate predictions on the validation split.
3. Search for the threshold maximizing F1-score independently for each label.
4. Save the resulting thresholds.
5. Use the validation-derived thresholds for subsequent test evaluation.

The optimized thresholds were:

label_caries                 : 0.45
label_gingivitis             : 0.25
label_malocclusion           : 0.35
label_pulpitis               : 0.35
label_tooth_loss             : 0.25
label_tooth_structure_loss   : 0.35

Thresholds were learned exclusively from the validation split.

The test split was not used for threshold selection.

Saved thresholds:

results/baseline/radiograph_only_6label/thresholds.json

---

# 12. Validation Results

Evaluation split:

Validation

Number of samples:

642

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1830 |
| Micro F1 | 0.4736 |
| Accuracy | 0.3894 |
| AUROC | 0.7003 |

## Optimized Label-specific Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.3097 |
| Micro F1 | 0.4716 |
| Accuracy | 0.2399 |
| AUROC | 0.7003 |

Threshold optimization substantially improved Macro F1:

0.1830 → 0.3097

while AUROC remained unchanged because AUROC is threshold-independent.

---

# 13. Test Results

Evaluation split:

Test

Number of samples:

642

Thresholds:

Learned exclusively from the validation split

## Default Threshold (0.5)

| Metric | Score |
|---|---:|
| Macro F1 | 0.1722 |
| Micro F1 | 0.4734 |
| Accuracy | 0.3801 |
| AUROC | 0.6988 |

## Optimized Validation-derived Thresholds

| Metric | Score |
|---|---:|
| Macro F1 | 0.3048 |
| Micro F1 | 0.4498 |
| Accuracy | 0.2399 |
| AUROC | 0.6988 |

Threshold optimization improved test Macro F1:

0.1722 → 0.3048

The optimized thresholds were not re-estimated on the test set.

Therefore, the optimized test result represents a leakage-safe evaluation using thresholds learned from validation data.

---

# 14. Generalization Analysis

The validation and test AUROC values are very close:

Validation AUROC:

0.7003

Test AUROC:

0.6988

Difference:

approximately 0.0015

This indicates very stable ranking performance between validation and test data.

Similarly, optimized Macro F1 is highly consistent:

Validation:

0.3097

Test:

0.3048

Difference:

approximately 0.0049

This close agreement suggests that the radiograph-only model generalizes reasonably well under the patient-level split.

---

# 15. Effect of Threshold Optimization

Threshold optimization had a substantial effect on Macro F1.

Validation:

0.1830 → 0.3097

Test:

0.1722 → 0.3048

The improvement is expected because the six diagnostic categories have different prevalence levels.

A single threshold of 0.5 is therefore suboptimal for the imbalanced multi-label setting.

However, threshold optimization reduced exact multi-label accuracy:

Validation:

0.3894 → 0.2399

Test:

0.3801 → 0.2399

This demonstrates that threshold selection should be interpreted according to the evaluation metric.

For this thesis, Macro F1 is particularly informative because it gives equal importance to all six diagnostic categories despite differences in prevalence.

---

# 16. Observations

## Radiographs Contain Useful Diagnostic Signal

The model achieved an AUROC of approximately 0.70 on both validation and test sets.

This indicates that radiographic images contain meaningful information for predicting the selected dental diagnoses.

---

## Stronger Behavior Than the Earlier 13-Label Baseline

Compared with the previous 13-label radiograph-only experiment, the six-label formulation produces substantially more stable and interpretable learning behavior.

The model begins learning useful classification signals from the early epochs instead of producing near-zero Macro F1 during the initial training phase.

This supports the decision to use the six-label benchmark as the main baseline classification setting.

---

## Macro F1 vs Micro F1

The difference between Macro F1 and Micro F1 is substantial.

For example, on the test set with the default threshold:

Macro F1:

0.1722

Micro F1:

0.4734

This indicates that performance is not evenly distributed across the six diagnostic categories.

The high Micro F1 is influenced more strongly by the more frequent labels, whereas Macro F1 exposes weaker performance on less frequent diagnoses.

Therefore, Macro F1 should remain an important primary metric for comparison across future baseline and multimodal experiments.

---

# 17. Limitations

This baseline has several limitations:

- Radiographs are naturally missing for a substantial proportion of visits in the COde dataset.
- Only visits containing radiographs can be evaluated.
- Mean pooling does not explicitly model spatial relationships or differences between radiograph types.
- The label reconstruction process is based on clinical information and may contain weak-label noise.
- The model uses only radiographic information and therefore cannot exploit complementary visual or textual information.
- The dataset is multi-label and imbalanced, making threshold-dependent metrics sensitive to threshold selection.

---

# 18. Role in Thesis

This experiment establishes the official six-label radiograph-only baseline for the COde dataset.

It provides a reference point for evaluating the contribution of other modalities and multimodal learning strategies.

The planned progression is:

1. Photograph-only baseline
2. Radiograph-only baseline
3. Text-only baseline
4. Image + Radiograph baseline
5. Image + Text baseline
6. Full multimodal baseline
7. Missing-modality / robust multimodal experiments

The main purpose of this experiment is not to achieve the final best performance.

Instead, it establishes how much diagnostic information can be extracted from radiographs alone and provides a reproducible baseline against which future multimodal and missing-modality approaches can be compared.

---

# 19. Final Baseline Result

The final official test performance of the six-label radiograph-only fine-tuned baseline is:

| Metric | Default Threshold | Optimized Thresholds |
|---|---:|---:|
| Macro F1 | 0.1722 | 0.3048 |
| Micro F1 | 0.4734 | 0.4498 |
| Accuracy | 0.3801 | 0.2399 |
| AUROC | 0.6988 | 0.6988 |

The optimized-threshold Macro F1 of:

0.3048

and AUROC of:

0.6988

are the key results to carry forward when comparing this baseline against subsequent experiments.

---

# 20. Reproducibility Artifacts

Six-label dataset:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

Model checkpoint:

results/baseline/radiograph_only_6label/best_model.pt

Optimized thresholds:

results/baseline/radiograph_only_6label/thresholds.json

Validation evaluation:

results/baseline/radiograph_only_6label/validation_evaluation.json

Test evaluation:

results/baseline/radiograph_only_6label/test_evaluation.json

---

# Conclusion

The six-label radiograph-only baseline successfully learned meaningful diagnostic representations from radiographic images alone.

The model achieved:

Test Macro F1:

0.3048

Test Micro F1:

0.4498

Test AUROC:

0.6988

when using validation-derived label-specific thresholds.

The close agreement between validation and test performance indicates stable generalization under the patient-level split.

This baseline is now established as the radiographic reference point for the next stages of the COde multimodal classification experiments.

**# Thesis Note 05.3**

**# Baseline Experiment C — Six-Label Photograph-only Fine-tuned Classification**

**## Overview**

This experiment represents the official photograph-only baseline for the six-label COde classification benchmark.

The objective is to establish a clean and reproducible unimodal baseline using intraoral photographs alone before introducing radiographs, clinical text, and multimodal fusion.

The experiment uses the previously constructed six-label patient-level benchmark and preserves the authoritative patient-level split.

The model receives only photographic images from each dental visit and predicts six reconstructed diagnostic categories.

This experiment establishes the visual photograph-based reference point for subsequent radiograph-only, text-only, and multimodal experiments.

\---

**# 1. Experiment Objective**

The goal of this experiment is to train and evaluate a photograph-only deep learning classifier for six-label dental diagnosis prediction.

The model receives only intraoral photographs from each patient visit and predicts the following six diagnostic categories:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth Loss
6. Tooth Structure Loss

The experiment answers the following research question:

> How effective are intraoral photographs alone for automated multi-label dental diagnosis classification on the six-label COde benchmark?

\---

**# 2. Six-Label Benchmark**

The experiment uses the finalized six-label patient-level benchmark derived from the previously reconstructed 13-label dataset.

The three original malocclusion categories:

- Class I Malocclusion
- Class II Malocclusion
- Class III Malocclusion

were combined into a single:

- Malocclusion

label.

The final label set is:

- label_caries
- label_gingivitis
- label_malocclusion
- label_pulpitis
- label_tooth_loss
- label_tooth_structure_loss

The resulting dataset artifact is:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

\---

**# 3. Dataset Configuration**

Dataset:

COde Dataset

Task:

Six-label multi-label dental diagnosis classification

Number of labels:

6

Total visits:

8,775

Total patients:

4,800

The six-label representation preserves the previously established patient-level split and does not introduce a new split.

\---

**# 4. Patient-Level Split**

The experiment uses the previously validated patient-level split.

Split strategy:

Patient-level split

Seed:

42

Train      : 3,360 patients / 6,129 visits

Validation :   720 patients / 1,330 visits

Test       :   720 patients / 1,316 visits

The patient-level split remains authoritative and was not modified during the six-label benchmark construction.

Previous leakage auditing confirmed that the split is leakage-safe.

The six-label construction changes only the label representation and does not introduce a new patient split.

\---

**# 5. Photograph Availability**

The COde dataset contains substantially more photographic information than radiographic information.

For the photograph-only experiment, only visits containing at least one photograph are retained by the modality requirement.

The resulting experiment-level sample counts are:

Train      : 6,126 samples

Validation : 1,330 samples

Test       : 1,316 samples

The difference between the total train visits (6,129) and photograph-only training samples (6,126) is caused by the modality requirement.

The photograph modality is therefore available for almost all visits in the patient-level benchmark.

\---

**# 6. Input Modality**

This experiment uses only intraoral photographs.

Used modality:

Photographs

Excluded modalities:

Radiographs

Clinical text

The input pipeline is:

Photographs

    |

    v

ResNet50 Encoder

    |

    v

Mean Feature Aggregation

    |

    v

Classification Head

    |

    v

Six-label Prediction

Only visits containing at least one photograph are included in the photograph-only experiment.

\---

**# 7. Model Architecture**

**## Image Encoder**

Backbone:

ResNet50

Initialization:

ImageNet pretrained weights

Training strategy:

Full encoder fine-tuning

The ResNet50 encoder parameters are updated during training rather than being frozen.

This allows the model to adapt the pretrained visual representation to the dental photograph domain.

\---

**# 8. Variable-Length Photograph Aggregation**

Each dental visit may contain a variable number of intraoral photographs.

The number of photographs per training visit varies considerably.

For the training split:

Total photographs:

34,726

Mean photographs per visit:

5.67

Median:

6

Maximum:

20

Each photograph is independently processed by the ResNet50 encoder and the resulting feature vectors are aggregated using mean pooling.

Architecture:

Photograph 1 ----\

Photograph 2 ----- ResNet50 ---- Mean Pooling ---- Classifier

Photograph N ----/

Mathematically:

z = (1/N) Σ f(x_i)

where:

- x_i represents the i-th photograph
- f(.) represents the ResNet50 encoder
- z represents the visit-level image representation
- N represents the number of photographs associated with the visit

This produces a fixed-dimensional visit-level representation regardless of the number of photographs.

\---

**# 9. Training Configuration**

Modality:

Photograph-only

Training samples:

6,126

Validation samples:

1,330

Batch size:

8

Number of epochs:

20

Loss function:

BCEWithLogitsLoss

Task formulation:

Multi-label classification

Learning rate:

1e-5

Weight decay:

1e-4

Encoder:

ResNet50

Pretrained:

ImageNet

Encoder:

Fine-tuned

Device:

CUDA

The batch size was reduced from the initial configuration because photograph visits contain substantially more images than radiograph visits, resulting in considerably higher GPU memory and computational cost per batch.

The final batch size of 8 provided a practical training configuration while preserving the same model architecture and optimization objective.

\---

**# 10. Training Results**

The model showed clear learning behavior during training.

At Epoch 1:

Train Loss:

0.3958

Validation Macro F1:

0.0488

Validation Micro F1:

0.1244

Validation AUROC:

0.6388

By Epoch 2, the model had already reached:

Validation Macro F1:

0.2479

Validation Micro F1:

0.4954

Validation AUROC:

0.6908

The model continued improving with substantial fluctuations in validation performance.

The best validation Macro F1 was obtained at:

Epoch 19

Best Validation Macro F1:

0.3773

Best Validation Micro F1:

0.4964

Best Validation Accuracy:

0.3940

Best Validation AUROC:

0.7013

Best Validation Loss:

0.9355

At Epoch 20, validation Macro F1 decreased to:

0.3402

while validation loss increased to:

1.0396.

This indicates that the model had begun to overfit the training data during the later stages of training.

The final best checkpoint was therefore associated with Epoch 19.

\---

**# 11. Best Model Checkpoint**

The best model was selected according to validation Macro F1.

Best epoch:

19

Best validation Macro F1:

0.37728554955632854

Best validation AUROC:

0.7013263491805887

The checkpoint was verified successfully and contains the expected ResNet50 model parameters for the six-label classification model.

Saved model:

results/baseline/photograph_only_6label/best_model.pt

The checkpoint was explicitly verified by loading the saved state dictionary into the photograph-only model architecture.

Checkpoint loading:

PASS

\---

**# 12. Threshold Optimization**

Because this is a multi-label classification problem with different label prevalences, a fixed probability threshold of 0.5 is not necessarily optimal for every diagnostic category.

Therefore, label-specific thresholds were optimized using the validation split.

The procedure was:

1. Train the model using the training split.
2. Select the best model according to validation Macro F1.
3. Generate predictions on the validation split.
4. Search for the threshold maximizing F1-score independently for each label.
5. Save the resulting label-specific thresholds.
6. Apply the validation-derived thresholds to the test split.

The final optimized thresholds were:

label_caries                : 0.35

label_gingivitis            : 0.50

label_malocclusion          : 0.40

label_pulpitis              : 0.40

label_tooth_loss            : 0.50

label_tooth_structure_loss  : 0.50

Thresholds were learned exclusively from the validation split.

The test split was not used for threshold selection.

Saved thresholds:

results/baseline/photograph_only_6label/thresholds.json

\---

**# 13. Validation Results**

Evaluation split:

Validation

Number of samples:

1,330

**## Default Threshold (0.5)**

| Metric | Score |
|---|---:|
| Macro F1 | 0.3773 |
| Micro F1 | 0.4964 |
| Accuracy | 0.3940 |
| AUROC | 0.7013 |

**## Optimized Label-specific Thresholds**

| Metric | Score |
|---|---:|
| Macro F1 | 0.3888 |
| Micro F1 | 0.4888 |
| Accuracy | 0.3549 |
| AUROC | 0.7013 |

Threshold optimization improved Validation Macro F1:

0.3773 → 0.3888

while AUROC remained unchanged because AUROC is independent of the classification threshold.

\---

**# 14. Test Results**

Evaluation split:

Test

Number of samples:

1,316

Thresholds:

Learned exclusively from the validation split

**## Default Threshold (0.5)**

| Metric | Score |
|---|---:|
| Macro F1 | 0.3801 |
| Micro F1 | 0.4679 |
| Accuracy | 0.3435 |
| AUROC | 0.6930 |

**## Optimized Validation-derived Thresholds**

| Metric | Score |
|---|---:|
| Macro F1 | 0.3812 |
| Micro F1 | 0.4606 |
| Accuracy | 0.3252 |
| AUROC | 0.6930 |

Threshold optimization improved Test Macro F1:

0.3801 → 0.3812

The optimized thresholds were not re-estimated on the test set.

Therefore, the optimized test result represents a leakage-safe evaluation using thresholds learned from the validation split.

\---

**# 15. Generalization Analysis**

The validation and test AUROC values are:

Validation AUROC:

0.7013

Test AUROC:

0.6930

Difference:

approximately 0.0083

The small difference indicates relatively stable ranking performance between validation and test data.

The Macro F1 values are also highly consistent:

Validation Macro F1:

0.3773

Test Macro F1:

0.3801

Difference:

approximately 0.0028

With optimized thresholds:

Validation:

0.3888

Test:

0.3812

Difference:

approximately 0.0076

The close agreement between validation and test performance suggests that the photograph-only model generalizes reasonably well under the patient-level split.

\---

**# 16. Effect of Threshold Optimization**

Threshold optimization produced a relatively small improvement in Macro F1 for the photograph-only model.

Validation:

0.3773 → 0.3888

Test:

0.3801 → 0.3812

Unlike the radiograph-only baseline, where threshold optimization produced a substantially larger improvement, the photograph-only model already achieved strong Macro F1 at the default threshold of 0.5.

This suggests that the probability calibration of the photograph-only model is more compatible with the default threshold for the six-label benchmark.

The optimized thresholds nevertheless remain the official threshold configuration because they are learned systematically from validation data and provide a label-specific decision rule.

\---

**# 17. Observations**

**## Photographs Contain Strong Diagnostic Signal**

The photograph-only model achieved an AUROC of:

0.6930

on the independent test set.

This demonstrates that intraoral photographs contain meaningful information for predicting the selected dental diagnoses.

The model also achieved:

Test Macro F1:

0.3801

with the default threshold.

This establishes photographs as a strong unimodal information source for the six-label benchmark.

\---

**## Strong Validation-Test Agreement**

The validation and test Macro F1 values are very close:

Validation:

0.3773

Test:

0.3801

The corresponding AUROC values are also close:

Validation:

0.7013

Test:

0.6930

This consistency is important because the test set was never used for model selection or threshold optimization.

The result therefore provides evidence of reasonable generalization under the patient-level split.

\---

**## Macro F1 vs Micro F1**

On the test set using the default threshold:

Macro F1:

0.3801

Micro F1:

0.4679

The higher Micro F1 indicates that performance is stronger when considering all label decisions collectively.

Macro F1 remains particularly important because it gives equal weight to each diagnostic category regardless of prevalence.

Therefore, Macro F1 should remain one of the primary metrics for comparing the photograph-only baseline with future radiograph-only and multimodal experiments.

\---

**## Comparison with Radiograph-only Baseline**

The photograph-only and radiograph-only baselines provide two complementary unimodal reference points.

Photograph-only test performance:

Macro F1:

0.3801

Micro F1:

0.4679

AUROC:

0.6930

Radiograph-only test performance:

Macro F1:

0.1722

Micro F1:

0.4734

AUROC:

0.6988

The photograph-only model achieves substantially higher Macro F1 than the radiograph-only model while achieving a very similar AUROC.

This suggests that photographs may provide more balanced diagnostic classification performance across the six labels, while radiographs retain strong ranking performance but exhibit weaker Macro F1 at the default threshold.

However, these results should not be interpreted as evidence that photographs are universally superior to radiographs.

The two modalities have different availability patterns, and radiograph-only evaluation is restricted to visits containing radiographs.

The final comparison of modality contributions will therefore be performed more rigorously in the subsequent multimodal experiments.

\---

\---

**# 18. Unimodal Baseline Comparison**

The two visual unimodal baselines provide complementary reference points for the six-label COde classification benchmark.

Both experiments use:

- The same six-label benchmark
- The same patient-level split
- The same random seed (42)
- The same ResNet50 ImageNet-pretrained backbone
- Full encoder fine-tuning
- Mean feature aggregation
- BCEWithLogitsLoss
- The same evaluation metrics

The primary difference is the input modality.

The photograph-only baseline uses intraoral photographs, whereas the radiograph-only baseline uses radiographic images.

**## Final Test Performance**

| Metric | Photograph-only | Radiograph-only |
|---|---:|---:|
| Macro F1 — Default Threshold | **0.3801** | 0.1722 |
| Macro F1 — Optimized Thresholds | **0.3812** | 0.3048 |
| Micro F1 — Default Threshold | 0.4679 | **0.4734** |
| Micro F1 — Optimized Thresholds | **0.4606** | 0.4498 |
| Accuracy — Default Threshold | 0.3435 | **0.3801** |
| Accuracy — Optimized Thresholds | 0.3252 | 0.2399 |
| AUROC | 0.6930 | **0.6988** |
| Test Samples | 1,316 | 642 |

\---

**## Macro F1 Comparison**

The photograph-only baseline substantially outperforms the radiograph-only baseline in Macro F1.

Using the default threshold:

Photograph-only:

0.3801

Radiograph-only:

0.1722

Difference:

approximately +0.2079

Using validation-derived optimized thresholds:

Photograph-only:

0.3812

Radiograph-only:

0.3048

Difference:

approximately +0.0764

This indicates that photographic images provide substantially more balanced classification performance across the six diagnostic categories under the current benchmark configuration.

The difference is particularly pronounced when using the default threshold of 0.5.

\---

**## Micro F1 Comparison**

The Micro F1 results are much closer between the two modalities.

Using the default threshold:

Photograph-only:

0.4679

Radiograph-only:

0.4734

Radiograph-only is approximately 0.0054 higher.

Using optimized thresholds:

Photograph-only:

0.4606

Radiograph-only:

0.4498

Photograph-only is approximately 0.0109 higher.

This indicates that the overall number of correct label decisions is relatively similar between the two modalities, despite the substantially different Macro F1 values.

The discrepancy between Macro F1 and Micro F1 suggests that modality performance is not evenly distributed across the six diagnostic categories.

\---

**## AUROC Comparison**

The AUROC values are also very similar:

Photograph-only:

0.6930

Radiograph-only:

0.6988

Difference:

approximately 0.0058

Radiographs achieve a slightly higher AUROC, indicating slightly stronger overall ranking performance.

However, the difference is small.

Therefore, both modalities contain meaningful predictive information for the six-label classification task.

The higher Macro F1 of the photograph-only model suggests that its advantage is primarily related to more balanced threshold-dependent classification performance rather than a large difference in overall ranking ability.

\---

**## Threshold Optimization Comparison**

Threshold optimization affects the two modalities differently.

| Modality | Default Macro F1 | Optimized Macro F1 | Improvement |
|---|---:|---:|---:|
| Photograph-only | 0.3801 | 0.3812 | +0.0011 |
| Radiograph-only | 0.1722 | 0.3048 | +0.1326 |

The radiograph-only model benefits substantially from label-specific threshold optimization.

In contrast, the photograph-only model already performs strongly using the default threshold of 0.5, and threshold optimization provides only a small additional improvement.

This difference suggests that the output probability distributions of the two unimodal models behave differently across the six diagnostic categories.

\---

**## Interpretation**

The unimodal comparison provides three important observations.

First, both modalities contain meaningful diagnostic information, as demonstrated by test AUROC values close to 0.70.

Second, photographs provide substantially stronger Macro F1 performance, particularly at the default threshold.

Third, radiographs achieve a slightly higher AUROC and similar Micro F1, indicating that radiographic representations still contain valuable complementary diagnostic information.

Therefore, the results do not suggest that one modality completely replaces the other.

Instead, they motivate the next stage of the thesis:

> If photographs and radiographs contain partially complementary diagnostic information, combining them may improve classification performance beyond either unimodal baseline.

This provides the direct motivation for the subsequent **Image + Radiograph multimodal baseline**.

\---

**## Baseline Comparison Summary**

| Aspect | Photograph-only | Radiograph-only |
|---|---|---|
| Input | Intraoral photographs | Radiographs |
| Test samples | 1,316 | 642 |
| Default Macro F1 | **0.3801** | 0.1722 |
| Optimized Macro F1 | **0.3812** | 0.3048 |
| Default Micro F1 | 0.4679 | **0.4734** |
| Optimized Micro F1 | **0.4606** | 0.4498 |
| Default Accuracy | 0.3435 | **0.3801** |
| Optimized Accuracy | **0.3252** | 0.2399 |
| AUROC | 0.6930 | **0.6988** |
| Main Strength | Balanced label-level performance | Strong ranking performance |
| Threshold Sensitivity | Low | High |

Overall, the photograph-only baseline currently provides the strongest unimodal Macro F1 result, while the radiograph-only baseline provides comparable AUROC and Micro F1 performance.

These complementary characteristics provide a strong motivation for evaluating multimodal fusion.

**# 19. Limitations**

This baseline has several limitations:

- Only photographic information is used.
- Clinical text and radiographic information are excluded.
- Mean pooling does not explicitly model spatial relationships between photographs.
- Mean pooling also treats all photographs equally despite potentially different diagnostic importance.
- The label reconstruction process is based on clinical information and may contain weak-label noise.
- The dataset is multi-label and imbalanced.
- Threshold-dependent metrics remain sensitive to the selected decision thresholds.
- The photograph-only experiment does not directly address naturally missing radiographs.

\---

**# 20. Role in Thesis**

This experiment establishes the official six-label photograph-only baseline for the COde dataset.

Together with the radiograph-only baseline, it provides the two primary unimodal visual reference points for the next stages of the thesis.

The planned progression is:

1. Photograph-only baseline
2. Radiograph-only baseline
3. Text-only baseline
4. Image + Radiograph baseline
5. Image + Text baseline
6. Full multimodal baseline
7. Missing-modality / robust multimodal experiments

The main purpose of this experiment is not to achieve the final best performance.

Instead, it quantifies the diagnostic information available from intraoral photographs alone and provides a reproducible reference against which multimodal and missing-modality approaches can be compared.

\---

**# 21. Final Baseline Result**

The final official test performance of the six-label photograph-only fine-tuned baseline is:

| Metric | Default Threshold | Optimized Thresholds |
|---|---:|---:|
| Macro F1 | 0.3801 | 0.3812 |
| Micro F1 | 0.4679 | 0.4606 |
| Accuracy | 0.3435 | 0.3252 |
| AUROC | 0.6930 | 0.6930 |

The key results to carry forward are:

Test Macro F1:

0.3801

Test Micro F1:

0.4679

Test AUROC:

0.6930

using the default threshold of 0.5.

The optimized-threshold Macro F1 is:

0.3812

with validation-derived label-specific thresholds.

\---

**# 22. Reproducibility Artifacts**

Six-label dataset:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json

Best model checkpoint:

results/baseline/photograph_only_6label/best_model.pt

Training history:

results/baseline/photograph_only_6label/history.json

Optimized thresholds:

results/baseline/photograph_only_6label/thresholds.json

Validation evaluation:

results/baseline/photograph_only_6label/validation_evaluation.json

Test evaluation:

results/baseline/photograph_only_6label/test_evaluation.json

\---

**# Conclusion**

The six-label photograph-only baseline successfully learned meaningful diagnostic representations from intraoral photographs alone.

The model achieved on the independent test set:

Test Macro F1:

0.3801

Test Micro F1:

0.4679

Test AUROC:

0.6930

with the default threshold of 0.5.

Using validation-derived label-specific thresholds produced:

Test Macro F1:

0.3812

Test Micro F1:

0.4606

Test AUROC:

0.6930

The close agreement between validation and test performance indicates stable generalization under the patient-level split.

The photograph-only baseline is now established as the official photographic reference point for the six-label COde classification benchmark.

The next baseline can therefore proceed to the next modality without changing the patient-level split or the six-label benchmark definition.

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

df = pd.read_csv(
    PROJECT_ROOT /
    "results" /
    "six_label_patient_level_dataset" /
    "labeled_dataset.csv"
)

print(df.shape)

(8775, 46)


In [3]:
# DELETE AFTER LEAKAGE AUDIT

import pandas as pd

text_cols = [
    "chief_complaint",
    "present_illness",
    "past_medical_record",
    "examination",
]

df["clinical_text"] = (
    df[text_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)


keywords = {
    "caries": [
        "caries",
        "carious",
        "decay",
    ],

    "gingivitis": [
        "gingivitis",
        "gingival inflammation",
    ],

    "pulpitis": [
        "pulpitis",
        "pulp",
    ],

    "tooth_loss": [
        "missing tooth",
        "tooth loss",
        "extracted",
    ],

    "malocclusion": [
        "malocclusion",
        "overbite",
        "crossbite",
    ],

    "structure_loss": [
        "fracture",
        "enamel",
        "dentin",
    ],
}


for label, words in keywords.items():

    print("\n==========", label, "==========")

    for w in words:

        count = (
            df["clinical_text"]
            .str.contains(
                w,
                regex=False
            )
            .sum()
        )

        print(
            f"{w:25s}: {count}"
        )


========== caries ==========
caries                   : 427
carious                  : 256
decay                    : 20

========== gingivitis ==========
gingivitis               : 1
gingival inflammation    : 253

========== pulpitis ==========
pulpitis                 : 1
pulp                     : 99

========== tooth_loss ==========
missing tooth            : 10
tooth loss               : 9
extracted                : 13

========== malocclusion ==========
malocclusion             : 7
overbite                 : 1474
crossbite                : 233

========== structure_loss ==========
fracture                 : 61
enamel                   : 37
dentin                   : 394


In [4]:
# DELETE AFTER LEAKAGE AUDIT

for col in text_cols:

    print("\n==============================")
    print(col)

    text = (
        df[col]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    for keyword in [
        "caries",
        "pulpitis",
        "gingivitis",
        "malocclusion",
        "fracture",
        "dentin",
        "enamel",
    ]:

        print(
            keyword,
            ":",
            text.str.contains(
                keyword,
                regex=False
            ).sum()
        )


chief_complaint
caries : 21
pulpitis : 1
gingivitis : 1
malocclusion : 2
fracture : 21
dentin : 0
enamel : 0

present_illness
caries : 24
pulpitis : 0
gingivitis : 0
malocclusion : 0
fracture : 17
dentin : 0
enamel : 0

past_medical_record
caries : 2
pulpitis : 0
gingivitis : 0
malocclusion : 1
fracture : 0
dentin : 0
enamel : 0

examination
caries : 416
pulpitis : 0
gingivitis : 0
malocclusion : 4
fracture : 47
dentin : 394
enamel : 37


Baseline Experiment D — Six-Label Clinical Text-only Fine-tuned Classification
Overview

This experiment represents the official clinical text-only baseline for the six-label COde classification benchmark.

The objective of this experiment is to quantify the diagnostic information contained in clinical textual records alone before introducing multimodal fusion.

The experiment uses the previously constructed six-label patient-level benchmark and preserves the authoritative patient-level split.

The model receives only clinical text extracted from dental visits and predicts six reconstructed diagnostic categories.

No photographic information or radiographic information is provided to the model.

This experiment establishes the textual unimodal reference point for subsequent multimodal experiments.

1. Experiment Objective

The goal of this experiment is to train and evaluate a clinical language model for six-label dental diagnosis prediction.

The model receives only textual clinical information from each dental visit and predicts the following six diagnostic categories:

Caries
Gingivitis
Malocclusion
Pulpitis
Tooth Loss
Tooth Structure Loss

The experiment answers the following research question:

How effective is clinical text alone for automated multi-label dental diagnosis classification on the six-label COde benchmark?

2. Six-Label Benchmark

The experiment uses the finalized six-label patient-level benchmark derived from the original reconstructed diagnostic labels.

The original three malocclusion categories:

Class I Malocclusion
Class II Malocclusion
Class III Malocclusion

were combined into a single:

Malocclusion

label.

The final label set is:

label_caries
label_gingivitis
label_malocclusion
label_pulpitis
label_tooth_loss
label_tooth_structure_loss

The dataset artifact remains:

results/six_label_patient_level_dataset/labeled_dataset.csv

Label distribution:

results/six_label_patient_level_dataset/label_split_distribution.csv

Dataset summary:

results/six_label_patient_level_dataset/dataset_summary.json
3. Dataset Configuration

Dataset:

COde Dataset

Task:

Six-label multi-label dental diagnosis classification

Number of labels:

6

Total visits:

8,775

Total patients:

4,800

The six-label benchmark preserves the previously validated patient-level split.

No new split was created during this experiment.

4. Patient-Level Split

The experiment uses the authoritative patient-level split generated during previous dataset construction.

Split strategy:

Patient-level split

Random seed:

42

Dataset partition:

Split	Patients	Visits
Train	3,360	6,129
Validation	720	1,330
Test	720	1,316

The split was not modified during the text-only experiment.

Previous leakage auditing confirmed that patient-level separation is preserved.

5. Clinical Text Availability Audit

Before training the text-only baseline, the availability and characteristics of clinical text were analyzed.

The clinical text source was constructed from dental visit textual records.

Total samples:

8775 visits

Empty text samples:

74

Mean text length:

384.45 characters

Median text length:

320 characters

Maximum text length:

1601 characters

Example clinical text:

The patient reports recurrent food impaction around tooth 48 for several years, causing intermittent discomfort. Over the past few years, the patient has experienced repeated food trapping at the site of tooth 48, prompting today’s evaluation...

The majority of visits contained meaningful clinical descriptions suitable for language-model training.

6. Text Input Construction

The clinical text representation was constructed from the following available fields:

chief_complaint
present_illness
past_medical_record
examination

These fields represent different aspects of dental clinical documentation:

chief_complaint

Contains the patient's main reason for visiting.

present_illness

Contains information about current symptoms and complaints.

past_medical_record

Contains relevant historical medical information.

examination

Contains clinical examination findings.

The final input sequence is a concatenation of available textual information from these fields.

7. Clinical Text Leakage Analysis

Because labels were reconstructed from clinical information, a dedicated leakage analysis was performed before training.

The objective was to determine whether the text input directly contains diagnostic keywords that could trivially reveal the target labels.

Keyword occurrence was evaluated across:

Entire clinical text
Individual text fields
7.1 Full Clinical Text Keyword Audit

Keyword frequencies in complete clinical text:

Caries
caries       : 427
carious      : 256
decay        : 20
Gingivitis
gingivitis              : 1
gingival inflammation   : 253
Pulpitis
pulpitis : 1
pulp      : 99
Tooth Loss
missing tooth : 10
tooth loss    : 9
extracted     : 13
Malocclusion
malocclusion : 7
overbite     : 1474
crossbite    : 233
Tooth Structure Loss
fracture : 61
enamel   : 37
dentin   : 394
7.2 Field-Level Leakage Analysis

Keyword analysis was also performed separately for:

chief_complaint
present_illness
past_medical_record
examination

The results showed that diagnostic terms are mainly concentrated in examination-related text.

Examples:

chief_complaint
caries       : 21
pulpitis     : 1
gingivitis   : 1
malocclusion : 2
fracture     : 21
present_illness
caries       : 24
fracture     : 17
examination

Contains the majority of diagnosis-related descriptions:

fracture : 47
dentin   : 394
enamel   : 37
7.3 Leakage Interpretation

The analysis indicates that clinical text naturally contains diagnostic information.

However, this does not represent artificial leakage.

The purpose of the text-only baseline is specifically to measure how much diagnostic information is available from routine clinical documentation.

The text modality therefore represents a realistic clinical scenario where dentists document findings that naturally correlate with diagnoses.

The result should not be interpreted as an image-independent diagnostic model replacing clinical assessment.

Instead, it establishes the upper information contribution of available textual documentation for later multimodal comparison.

8. Model Architecture
Text Encoder

The model uses:

DistilBERT

Pretrained model:

distilbert-base-uncased

The pretrained language model was fine-tuned on the COde dental clinical text.

Tokenization

Tokenizer:

DistilBERT tokenizer

Maximum sequence length:

256 tokens

Each sample produces:

input_ids

attention_mask

labels
Classification Head

The final hidden representation from DistilBERT is passed into a multi-label classification head.

Architecture:

Clinical Text

      |

      v

DistilBERT Encoder

      |

      v

Text Representation

      |

      v

Linear Classification Head

      |

      v

Six Diagnostic Labels

9. Training Configuration

Modality:

Clinical Text-only

Training samples:

6129

Validation samples:

1330

Test samples:

1316

Batch size:

16

Number of epochs:

20

Loss function:

Task formulation:

Multi-label classification

Optimizer:

Fine-tuning optimization of DistilBERT parameters

Pretrained model:

Device:

CUDA

The text-only model was trained using the same evaluation protocol as the visual baselines.

The same six-label benchmark and patient-level split were preserved to ensure fair comparison across modalities.

10. Training Results

The model demonstrated strong learning behavior from the beginning of training.

At Epoch 1:

Training Loss:

0.1868

Validation performance:

Metric	Score
Macro F1	0.7553
Micro F1	0.8605
Accuracy	0.8083
AUROC	0.9639

The model continued improving during early epochs.

Best Validation Performance

The best validation Macro F1 was obtained at:

Epoch 14

Best validation metrics:

Metric	Score
Macro F1	0.8472
Micro F1	0.9003
Accuracy	0.8579
AUROC	0.9771

The corresponding checkpoint was selected as the official text-only model.

Training Behavior

The training loss continued decreasing:

Epoch 1:

0.1868

Epoch 20:

0.0070

However, after approximately epoch 14, validation metrics started fluctuating.

This indicates that the model reached its optimal generalization point before the end of training.

The final checkpoint was therefore selected according to validation Macro F1 rather than the final training epoch.

11. Best Model Checkpoint

Model selection criterion:

Validation Macro F1

Best epoch:

14

Best validation Macro F1:

0.8472472516080712

Checkpoint:

results/baseline/text_only_6label/best_model.pt

The checkpoint was verified successfully.

Verification:

PASS

Stored information:

model_state
best_f1
epoch

Checkpoint metadata:

epoch:
14

best_f1:
0.8472472516080712
12. Threshold Optimization

Because this experiment is formulated as a multi-label classification problem, each diagnostic category may require a different decision threshold.

Therefore, label-specific thresholds were optimized using the validation split.

The procedure was:

Load the best validation checkpoint.
Generate validation probabilities.
Search threshold values independently for each label.
Select the threshold maximizing label-level F1-score.
Save thresholds.
Apply the validation-derived thresholds to the test set.

Threshold optimization was performed without accessing test labels.

Optimized Thresholds

The final validation-derived thresholds were:

Label	Threshold
Caries	0.90
Gingivitis	0.30
Malocclusion	0.45
Pulpitis	0.50
Tooth Loss	0.50
Tooth Structure Loss	0.25

Saved artifact:

results/baseline/text_only_6label/thresholds.json
13. Validation Results

Evaluation split:

Validation

Samples:

1330
Default Threshold (0.5)
Metric	Score
Macro F1	0.8472
Micro F1	0.9003
Accuracy	0.8579
AUROC	0.9771
Optimized Thresholds

The optimized validation metrics were not used for model selection because thresholds are derived from the same validation split.

The validation thresholds were only used for independent test evaluation.

14. Test Results

Evaluation split:

Test

Samples:

1316

Thresholds:

Derived exclusively from validation split.

Default Threshold (0.5)
Metric	Score
Macro F1	0.8404
Micro F1	0.8899
Accuracy	0.8397
AUROC	0.9709
Optimized Validation-derived Thresholds
Metric	Score
Macro F1	0.8382
Micro F1	0.8872
Accuracy	0.8351
AUROC	0.9709

The optimized thresholds produced a very small decrease compared with the default threshold.

Therefore, the default threshold of 0.5 remains the primary reported result for this baseline.

15. Generalization Analysis

The validation and test performance were highly consistent.

AUROC Comparison

Validation:

0.9771

Test:

0.9709

Difference:

≈ 0.0062
Macro F1 Comparison

Validation:

0.8472

Test:

0.8404

Difference:

≈ 0.0068

The small performance gap indicates strong generalization under the patient-level split.

The model was never evaluated on patients appearing in the training data.

16. Effect of Threshold Optimization

Threshold optimization had minimal effect on the text-only model.

Comparison:

Setting	Macro F1
Default threshold	0.8404
Optimized thresholds	0.8382

Unlike the radiograph-only baseline, where threshold optimization significantly improved Macro F1, the text-only model already produced well-calibrated probabilities under the default threshold.

This suggests that the strong textual signal allowed the model to separate positive and negative samples effectively without aggressive threshold adjustment.

17. Observations
Clinical Text Contains Strong Diagnostic Signal

The text-only baseline achieved:

Test Macro F1:

0.8404

Test AUROC:

0.9709

These results demonstrate that clinical documentation contains substantial diagnostic information for the selected six-label benchmark.

The performance is considerably higher than the visual-only baselines, indicating that textual records provide a strong modality for dental diagnosis prediction.

Strong Validation-Test Agreement

The close agreement between validation and test performance indicates robust generalization.

Metric	Validation	Test
Macro F1	0.8472	0.8404
AUROC	0.9771	0.9709

The small difference confirms that the model benefits from the patient-level split and does not rely on patient overlap.

Macro F1 vs Micro F1

Test performance:

Macro F1:

0.8404

Micro F1:

0.8899

The difference between Macro and Micro F1 is relatively small compared with the visual baselines.

This suggests that the text-only model performs consistently across the six diagnostic categories despite label imbalance.